# 01 - GPU diagnostics

Checks whether this runtime can host CatVTON **inference** (about 8 GB VRAM in bf16 at 1024x768) and
**LoRA fine-tuning at 512x384** (estimated 6-10 GB with fp16 base weights, gradient checkpointing, batch 1).
The training estimate is NOT verified until notebook 03's one-step smoke test reports a measured peak.

Use *Runtime -> Change runtime type -> T4 GPU* (Colab) or *Accelerator -> GPU* (Kaggle). Nothing here
installs or downloads model weights.

In [ ]:
!nvidia-smi

In [ ]:
import shutil, subprocess, sys
import torch

print('python        :', sys.version.split()[0])
print('torch         :', torch.__version__)
print('torch CUDA    :', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

report = {'cuda_available': torch.cuda.is_available()}
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    report.update(gpu=p.name, vram_gb=round(p.total_memory / 1024**3, 1),
                  compute_capability=f'{p.major}.{p.minor}', bf16_supported=torch.cuda.is_bf16_supported())
    print('GPU           :', p.name)
    print('VRAM          : %.1f GB' % report['vram_gb'])
    print('compute cap.  :', report['compute_capability'])
    print('bf16 supported:', report['bf16_supported'], '(T4 lacks native bf16: use fp16)')
else:
    print('\nNo GPU. Change the runtime type before running notebooks 02 and 03.')

total, used, free = shutil.disk_usage('/content')
report['disk_free_gb'] = round(free / 1024**3, 1)
print('disk free     : %.1f GB (/content)' % report['disk_free_gb'])

In [ ]:
# Verdict (thresholds are rules of thumb, not guarantees)
def verdict(r):
    if not r['cuda_available']:
        return 'NO GPU: inference and training cannot run here.'
    v = r['vram_gb']
    out = []
    out.append('inference (bf16/fp16, 1024x768): ' + ('likely OK' if v >= 10 else 'marginal / may fail'))
    out.append('LoRA fine-tune 512x384, batch 1  : ' + ('likely OK' if v >= 14 else 'marginal - try batch 1, 8-bit Adam' if v >= 10 else 'NOT recommended (<10 GB)'))
    if r['disk_free_gb'] < 15:
        out.append('disk: <15 GB free - base weights (~4 GB) + caches may not fit')
    return '\n'.join(out)

print(verdict(report))
report